# Validate & Fill Gadgets with Audit and GadgetProfile

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do you mathematically verify that a gadget implements its target logical instruction, and how does `ec.filled` synthesize missing parity checks and readouts?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Compare `GadgetProfile(gadget).objective` against the computed circuit action `GadgetProfile(gadget).action`.
- Run `ec.audit(protocol)` to detect missing readouts, invalid parity checks, or mismatched circuit actions.
- Use `ec.filled(gadget)` to automatically synthesize missing parity `checks` and logical `readouts`.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## How `ec.GadgetProfile` Verifies a Gadget: Outcome-Complete Simulation & Circuit Action

How do we know whether a gadget circuit + readout/check annotations actually implements its declared logical instruction?
`ec.GadgetProfile(gadget)` performs **outcome-complete stabilizer simulation** (powered by `paulimer`) and unfolds the gadget into its logical **circuit action** (`profile.action`), then compares it against the instruction's declared **`profile.objective`**:
- **`profile.objective`**: The ideal stabilizer channel specified by `gadget.implements` on the boundary codes.
- **`profile.action`**: The actual stabilizer channel realized by running `gadget.circuit`, projecting onto `gadget.checks`, and extracting `gadget.readouts`.
- **`profile.action.is_equivalent_to(profile.objective)`**: Returns `True` if and only if the gadget faithfully implements the logical instruction across all inputs and measurement branches!

## Whole-Protocol Verification with `ec.audit(protocol)`

Calling `ec.audit(protocol)` validates every layer, code, instruction, and gadget in a `Qodec` protocol in a single call.

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Inspect `GadgetProfile.objective` vs `GadgetProfile.action` & Run `ec.audit`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

cx_profile = ec.GadgetProfile(protocol.layers[0].gadgets["cx_all"])
print("=== GadgetProfile(cx_all) ===")
print("Objective :", cx_profile.objective)
print("Action    :", cx_profile.action)
print("Equivalent?", cx_profile.action.is_equivalent_to(cx_profile.objective))

print("\n=== Whole-Protocol Audit ===")
print(ec.audit(protocol))

=== GadgetProfile(cx_all) ===
Objective : mapping:
  X_0 → X_0 X_2
  Z_0 → Z_0
  X_1 → X_1 X_3
  Z_1 → Z_1
  X_2 → X_2
  Z_2 → Z_0 Z_2
  X_3 → X_3
  Z_3 → Z_1 Z_3
Action    : mapping:
  X_0 → X_0 X_2
  Z_0 → Z_0
  X_1 → X_1 X_3
  Z_1 → Z_1
  X_2 → X_2
  Z_2 → Z_0 Z_2
  X_3 → X_3
  Z_3 → Z_1 Z_3
Equivalent? True

=== Whole-Protocol Audit ===


audit: ok


## Diagnosing Broken Gadgets & Automatic Synthesis with `ec.filled`

What happens when a gadget designer writes a new circuit (or modifies an existing one) without yet knowing the exact parity readout and check formulas?
Let's test this on `measure_z_all` and `syndrome`:
1. If we clear `measure_z_all.readouts = []`, `ec.audit` catches `error: gadget/missing-observable` and `objective.why_not_equivalent_to(action)` pinpoints the missing observables (`observe[0]`, `observe[1]`).
2. If we clear `syndrome.checks = []`, `ec.audit` catches `warning: gadget/incomplete-output-frame`.
3. Calling **`ec.filled(gadget)`** automatically solves the Pauli transformation through `gadget.circuit` and populates the missing `readouts` and `checks`!

In [3]:
# Diagnose Missing Readouts/Checks with `ec.audit` & Repair with `ec.filled`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

broken_protocol = qodec.Qodec.loads(protocol.dumps())
broken_mz = broken_protocol.layers[0].gadgets["measure_z_all"]
broken_syn = broken_protocol.layers[0].gadgets["syndrome"]

# 1. Break `measure_z_all` readouts and `syndrome` checks
broken_mz.readouts = []
broken_syn.checks = []

print("=== Audit of Intentionally Broken Protocol ===")
print(ec.audit(broken_protocol))

mz_prof = ec.GadgetProfile(broken_mz)
print("=== Why is broken `measure_z_all` not equivalent to its objective? ===")
print(mz_prof.objective.why_not_equivalent_to(mz_prof.action))

# 2. Automatically synthesize the missing readouts and checks with `ec.filled`!
repaired_protocol = ec.filled(broken_protocol)
print("\n=== Audit After `ec.filled(broken_protocol)` ===")
print(ec.audit(repaired_protocol))
print("Synthesized measure_z_all readouts:", repaired_protocol.layers[0].gadgets["measure_z_all"].readouts)
print("Synthesized syndrome checks       :", repaired_protocol.layers[0].gadgets["syndrome"].checks)

=== Audit of Intentionally Broken Protocol ===


[ERROR] gadget/missing-observable
layers[0].gadgets['measure_z_all'] (C4 -> stim)
readouts[0] has no equation for the required logical Z_0 measurement
    Verified readout equation: ["in[0].z[0]", "circuit.readouts[0]", "circuit.readouts[2]"]

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_z_all'] (C4 -> stim)
readouts[1] has no equation for the required logical Z_1 measurement
    Verified readout equation: ["in[0].z[1]", "circuit.readouts[0]", "circuit.readouts[1]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['syndrome'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[0] (X_0 X_1 X_2 X_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[0]", "in[0].stabilizers[0]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['syndrome'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[1] (Z_0 Z_1 Z_2 Z_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified


=== Audit After `ec.filled(broken_protocol)` ===


audit: ok
Synthesized measure_z_all readouts: (Readout(position=0, name=None, is_flag=False, equation=['in[0].z[0]', 'circuit.readouts[0]', 'circuit.readouts[2]']), Readout(position=1, name=None, is_flag=False, equation=['in[0].z[1]', 'circuit.readouts[0]', 'circuit.readouts[1]']))
Synthesized syndrome checks       : ((Reference('circuit.readouts[0]'), Reference('in[0].stabilizers[0]')), (Reference('circuit.readouts[1]'), Reference('in[0].stabilizers[1]')), (Reference('circuit.readouts[0]'), Reference('out[0].stabilizers[0]')), (Reference('circuit.readouts[1]'), Reference('out[0].stabilizers[1]')))


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What does <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">ec.audit(protocol)</code> report when a measurement gadget is missing its classical observable readout formula versus when a gadget is missing its stabilizer parity checks?</summary>

<div style="padding:0.1em 1em;">

Missing observable readouts produce `error: gadget/missing-observable` (because the gadget fails to produce the declared logical measurement output), while missing stabilizer checks produce `warning: gadget/incomplete-output-frame` (because the output code stabilizers are left unconstrained).

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;When can <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">ec.filled(gadget)</code> automatically derive <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">checks</code> and <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">readouts</code>, and when do you still need to provide <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">readouts</code> manually?</summary>

<div style="padding:0.1em 1em;">

`ec.filled` can automatically derive stabilizer `checks` and logical observable `readouts` from the boundary codes and circuit Pauli transformation. However, if a gadget introduces a custom post-selection `flag` (such as `flags = ['reject']`), you must specify which classical readout bit corresponds to `reject` in `gadget.readouts`, because a flag is a design choice rather than a code stabilizer.

</div>

</details></div>

## Exercise: Synthesize Missing Parity Checks with ec.filled

Use `ec.filled(syn)` to reconstruct the cleared parity checks on the `syndrome` gadget and return the number of synthesized checks.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def repair_syndrome_checks() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    syn.checks = []
    # Use ec.filled(syn) to restore the parity checks and return len(repaired.checks)
    return ...

**Hint**

Call `repaired = ec.filled(syn)` and return `len(repaired.checks)`.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def repair_syndrome_checks() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    syn.checks = []
    repaired = ec.filled(syn)
    return len(repaired.checks)

`ec.filled(syn)` uses outcome-complete stabilizer simulation to synthesize all `4` parity checks for the `syndrome` gadget (`2` input-to-readout stabilizer checks and `2` input-to-output stabilizer boundary checks).

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 4
- **Learning Objectives:**
  1. Compare `profile.objective` and `profile.action` using `ec.GadgetProfile(gadget)`.
  2. Run `ec.audit(protocol)` to verify a clean protocol (`audit: ok`).
  3. Intentionally break `readouts` and `checks` on a gadget, diagnose the resulting `gadget/missing-observable` error and `gadget/incomplete-output-frame` warning using `ec.audit` and `objective.why_not_equivalent_to(action)`, and repair the gadget automatically using `ec.filled(gadget)`.
- **Socratic Prompt:** When the learner clears `syndrome.checks = []` in Cell 2 below, ask: *"Why did `ec.audit` report `incomplete-output-frame` on the OUTPUT stabilizers when we removed the syndrome checks?"* (Answer: Without the syndrome measurement checks, the stabilizer generator signs on the output block are no longer fixed relative to the input block!)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../03-gadget-anatomy/gadget_anatomy.workbook.ipynb" title="Previous unit: Inspect Gadget Circuits, Checks & Readouts" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Inspect Gadget Circuits, Checks & Readouts</span></span></a><a href="../05-circuit-distance/circuit_distance.workbook.ipynb" title="Next unit: Characterize Circuit Distance & Problematic Errors" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Characterize Circuit Distance & Problematic Errors</span></span><span aria-hidden="true">&#8250;</span></a></div>